# Overview of anomalies

In [14]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [16]:
se_cols = [
    'mse',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_97',
    # 'mse_95',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

se_rank_cols = [
    f"rank_{col}" for col in se_cols
]

# ----------------------------------------------
rse_cols = [
    f"{col}_rel" for col in se_cols
]

rse_rank_cols = [
    f"rank_{col}_rel" for col in se_cols
]
# ----------------------------------------------
se_family = [
    'mse',
    'mse_97',
    # 'mse_95',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

rse_family = [
    f'{col}_rel' for col in se_family
]

# Custom functions

## IDs top anomalies

In [17]:
def get_ids(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [18]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    ids_top_list = []

    for score in scores_list:

        ids_set = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

        ids_top_dict[score] = ids_set

        ids_top_list += list(ids_set)


    n_dictinct_top = len(set(ids_top_list))

    print(f"N unique: {n_dictinct_top}")


    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_dictinct_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.4f}%")

    return unique_ids_dict, ids_top_dict

In [19]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}
    ids_top_list = []


    for score in scores_list:

        ids_set = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

        ids_top_dict[score] = ids_set

        ids_top_list += list(ids_set)

    n_dictinct_top = len(set(ids_top_list))
    print(f"N unique: {n_dictinct_top}")

    common_ids_set = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_set)
    common_pct = n_common/n_dictinct_top*100 
    print(f"N common:\n{n_common} -- > {common_pct:4f}%")

    return common_ids_set, ids_top_dict

## Figures

In [20]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [21]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_id = 'bin_00'
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"
os.makedirs(f"{ch_4_dir}/{bin_id}", exist_ok=True)

## Data

In [22]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [23]:
score_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)

rank = np.arange(score_df.shape[0]) + 1
score_rank_df = score_df.copy()
# score_rank_df
for col in score_df.columns:

    index_sorted = score_df.sort_values(
        by=col, ascending=False
    ).index

    score_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_rank_df[f'rank_{col}'].astype(int)

n_spec = score_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [24]:
score = 'mse_97'
score_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(10)

,mse_97,rank_mse_97
specobjid,,
2934148553238407168,64.415439,1.0
2009786658647468032,27.697053,2.0
1079775709385222144,25.091922,3.0
407585970297268224,19.874186,4.0
1988492400661653504,18.663280,5.0
352582071014680576,18.491562,6.0
574383539907225600,16.363535,7.0
1721507603789408256,16.334563,8.0
827663192506263552,16.237491,9.0


# Figs top anomalies

In [25]:
# # ```python
# n_top = 1000
# all_scores = se_cols + rse_cols
# plt.ioff()

# for score in all_scores:

#     specids_top_1 = score_df[score].sort_values(
#         ascending=False
#     ).index.to_numpy()[:n_top]

#     ranks = np.zeros(n_top).astype(int)

#     specs_top_1 = np.empty((n_top, wave.size))

#     for i, objid in enumerate(specids_top_1):

#         spec_idx = specobjid_to_idx(
#             objid, idx_id_spec
#         )

#         specs_top_1[i, :] = spectra[spec_idx, :]

#         ranks[i] = i

#     # -----------------------------------------------------------

#     save_to = f"{scores_dir}/{bin_id}/figs/{score}"

#     os.makedirs(save_to, exist_ok=True)

#     anomaly_plot(
#         wave_nm, specs=specs_top_1,
#         objids=specids_top_1, ranks=ranks,
#         save_to=save_to
#     )
# # ```

# No free lunch theorem

## IDs per score

In [26]:
ids_top_dict = {}

all_scores = se_cols + rse_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

n_top_1 = len(ids_top_dict[score])
n_top_1

1819

# Distinct IDs

## SE family

In [27]:
se_unique_ids_dict, se_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 2906
Unique to mse:
322 --> 11.0805%
Unique to mse_filter_250:
97 --> 3.3379%
Unique to mse_97:
39 --> 1.3421%
Unique to mse_filter_250_97:
70 --> 2.4088%


In [15]:
score = 'mse'
unique_score_ids = list(se_unique_ids_dict[score])
score_rank_df.loc[
    unique_score_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
1475068566825887744,61.730736,50.0
1920900747541637120,59.656955,54.0
797248221209454592,55.961233,61.0
570855206623930368,55.472211,62.0
422244936562272256,52.969322,71.0


### Figs unique per SE

In [16]:
plt.ioff()

for score in se_cols:

    specids = list(se_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_se/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## RSE family

In [28]:
rse_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=rse_cols,
    quantile=99,
    # n_top=1000, use_ntop=True
)

N unique: 2363
Unique to mse_rel:
132 --> 5.5861%
Unique to mse_filter_250_rel:
56 --> 2.3699%
Unique to mse_97_rel:
38 --> 1.6081%
Unique to mse_filter_250_97_rel:
53 --> 2.2429%


In [18]:
score = 'mse_rel'
unique_res_ids = list(rse_unique_ids_dict[score])
score_rank_df.loc[
    unique_res_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse_rel,rank_mse_rel
specobjid,,
2418585633070016512,77.904882,2.0
1475068566825887744,56.967304,4.0
694807820633139200,47.212674,8.0
2510773073792231424,45.930231,10.0
570855206623930368,45.189434,11.0


### Figs unique per RES

In [19]:
plt.ioff()

for score in rse_cols:

    specids = list(rse_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_rse/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## All scores

In [29]:
all_scores = se_cols + rse_cols

all_unique_ids_dict, all_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=all_scores,
    quantile=99,
    # n_top=1000, use_ntop=True
)

N unique: 3229
Unique to mse:
227 --> 7.0300%
Unique to mse_filter_250:
60 --> 1.8582%
Unique to mse_97:
13 --> 0.4026%
Unique to mse_filter_250_97:
10 --> 0.3097%
Unique to mse_rel:
11 --> 0.3407%
Unique to mse_filter_250_rel:
25 --> 0.7742%
Unique to mse_97_rel:
23 --> 0.7123%
Unique to mse_filter_250_97_rel:
38 --> 1.1768%


In [21]:
score = 'mse'
unique_all_ids = list(all_unique_ids_dict[score])
score_rank_df.loc[
    unique_all_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
422244936562272256,52.969322,71.0
1247606277450786816,45.161588,95.0
518076717147908096,40.807039,117.0
1527901749824743424,40.380793,121.0
951385736106502144,37.902900,140.0


### Figs unique among all

In [22]:
all_scores = se_cols + rse_cols
plt.ioff()

for score in all_scores:

    specids = list(all_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_all/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

# Common IDs

## SE family

In [30]:
se_common_ids, se_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 2906
N common:
847 -- > 29.146593%


In [24]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]
common_se_ids = list(se_common_ids)
score_rank_df.loc[
    common_se_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
407585970297268224,241.576758,33.340267,19.874186,19.047799,1.0,20.0,4.0,5.0
407580747617036288,218.632851,32.058104,14.698924,14.882089,2.0,23.0,25.0,31.0
2027890112850847744,194.072076,179.054097,14.479340,14.381004,3.0,1.0,32.0,63.0
1079775709385222144,130.322662,46.050367,25.091922,26.981096,7.0,9.0,3.0,2.0
1951274199876134912,112.283933,40.480495,13.495517,13.641587,10.0,12.0,112.0,207.0


### Figs common among SE

In [25]:
specids = list(se_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_ses"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## RSE Family

In [31]:
rse_common_ids, res_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=rse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 2363
N common:
1349 -- > 57.088447%


In [27]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_rse_ids = list(rse_common_ids)

score_rank_df.loc[
    common_rse_ids, scores + rank_scores
].sort_values(by='mse_rel', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
2027890112850847744,194.072076,179.054097,14.479340,14.381004,3.0,1.0,32.0,63.0
2934148553238407168,81.470618,72.617946,64.415439,69.219490,23.0,3.0,1.0,1.0
1579720636568201216,57.900874,52.794826,14.677076,13.961484,59.0,5.0,26.0,122.0
1230680384782493696,74.719197,52.622995,13.576213,13.968721,29.0,6.0,98.0,121.0
1079775709385222144,130.322662,46.050367,25.091922,26.981096,7.0,9.0,3.0,2.0


### Figs common among RSE

In [28]:
specids = list(rse_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse_rel'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_rses/"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## All scores

In [32]:
all_scores = se_cols + rse_cols 
all_common_ids, all_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3229
N common:
737 -- > 22.824404%


In [30]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_all_ids = list(all_common_ids)
score_rank_df.loc[
    common_all_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head(10)

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
407585970297268224,241.576758,33.340267,19.874186,19.047799,1.0,20.0,4.0,5.0
407580747617036288,218.632851,32.058104,14.698924,14.882089,2.0,23.0,25.0,31.0
2027890112850847744,194.072076,179.054097,14.479340,14.381004,3.0,1.0,32.0,63.0
1079775709385222144,130.322662,46.050367,25.091922,26.981096,7.0,9.0,3.0,2.0
1951274199876134912,112.283933,40.480495,13.495517,13.641587,10.0,12.0,112.0,207.0
796155581562906624,103.215549,21.099387,13.444095,13.460088,13.0,68.0,116.0,271.0
2009786658647468032,87.562080,28.517228,27.697053,24.646343,20.0,29.0,2.0,3.0
2934148553238407168,81.470618,72.617946,64.415439,69.219490,23.0,3.0,1.0,1.0
1230680384782493696,74.719197,52.622995,13.576213,13.968721,29.0,6.0,98.0,121.0


### Figures

In [31]:
all_scores = se_cols + rse_cols
plt.ioff()


specids = list(all_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, 'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_all"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)